In [1]:
# ==== 0) Install & imports ====
import os, io, base64, json, re, time, pathlib
from typing import Dict, Any, List
from PIL import Image
import pandas as pd
from tqdm import tqdm

In [ ]:
# Robust finder: locate ".../Project/PicturesAlbum" anywhere, fuzzy names, then mirror locally
!pip -q install --upgrade google-api-python-client

import io, re
from pathlib import Path
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
drive = build("drive", "v3")

def normalize(s):  # ignore case, spaces, punctuation/accents
    import unicodedata
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\W_]+", "", s).casefold()

TARGET_PROJECTS = {"project", "projects", "proyecto"}
TARGET_PICTURES  = {"picturesalbum", "pictures album", "albumdefotos", "fotosalbum", "album", "pictures"}

def list_folders(query):
    out, token = [], None
    while True:
        resp = drive.files().list(
            q=f"mimeType='application/vnd.google-apps.folder' and trashed=false and ({query})",
            fields="files(id,name,parents,driveId), nextPageToken",
            pageSize=200,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
            pageToken=token
        ).execute()
        out += resp.get("files", [])
        token = resp.get("nextPageToken")
        if not token: break
    return out

def list_children_folders(parent_id):
    out, token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false",
            fields="files(id,name), nextPageToken",
            pageSize=200,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
            pageToken=token
        ).execute()
        out += resp.get("files", [])
        token = resp.get("nextPageToken")
        if not token: break
    return out

def list_images(parent_id):
    out, token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{parent_id}' in parents and mimeType contains 'image/' and trashed=false",
            fields="files(id,name,mimeType), nextPageToken",
            pageSize=200,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
            pageToken=token
        ).execute()
        out += resp.get("files", [])
        token = resp.get("nextPageToken")
        if not token: break
    return out

def download_file(file_id, dst_path: Path):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    request = drive.files().get_media(fileId=file_id)
    with io.FileIO(str(dst_path), "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

# 1) Find candidate "Project" folders anywhere
projects = list_folders("name contains 'Project' or name contains 'Proyecto'")
projects_norm = [(p, normalize(p["name"])) for p in projects]
proj_hits = [p for p, n in projects_norm if n in TARGET_PROJECTS]
if not proj_hits:
    # fallback: accept any folder whose normalized name contains 'project'
    proj_hits = [p for p, n in projects_norm if "project" in n or "proyecto" in n]

if not proj_hits:
    raise RuntimeError("No folder resembling 'Project' found anywhere. Please check the exact name in Drive.")

# 2) For each candidate Project, look for a child that matches PicturesAlbum (fuzzy)
picked_project = None
picked_pictures = None
all_children_preview = []
for pr in proj_hits:
    kids = list_children_folders(pr["id"])
    names = [k["name"] for k in kids]
    all_children_preview.append((pr["name"], names))
    # fuzzy match photos folder
    match = None
    for k in kids:
        kn = normalize(k["name"])
        if kn in TARGET_PICTURES or any(t in kn for t in TARGET_PICTURES):
            match = k
            break
    if match:
        picked_project = pr
        picked_pictures = match
        break

if not picked_pictures:
    # Print what we saw to help verify names
    print("Couldn't auto-match the pictures folder. Children we saw under candidate 'Project' folders:")
    for pname, kids in all_children_preview:
        print(f" - {pname}: {kids}")
    # Last resort: search globally for a Pictures folder by name and show options
    pics_global = list_folders("name contains 'Pictures' or name contains 'Album' or name contains 'Fotos'")
    print("\nGlobal candidates for 'PicturesAlbum' (pick the right one by updating TARGET_PICTURES set above if needed):")
    for f in pics_global[:20]:
        print(f"  {f['name']}  (id={f['id']})")
    raise RuntimeError("Did not find a child like 'PicturesAlbum' under any 'Project'. See printed lists above.")

print(f"Using Project: {picked_project['name']}  | Pictures folder: {picked_pictures['name']}")

# 3) Mirror all teammate subfolders from PicturesAlbum
team_folders = list_children_folders(picked_pictures["id"])
if not team_folders:
    raise RuntimeError("No subfolders found inside the pictures folder.")

LOCAL_MIRROR = Path("/content/PicturesAlbum_Mirror")
downloaded = 0
for f in team_folders:
    imgs = list_images(f["id"])
    print(f"Syncing {f['name']} ({len(imgs)} images)")
    for g in imgs:
        dst = LOCAL_MIRROR / f["name"] / g["name"]
        if not dst.exists():
            download_file(g["id"], dst)
            downloaded += 1

print(f"Mirror complete. Downloaded {downloaded} new files.")
ROOT = LOCAL_MIRROR
print("ROOT set to:", ROOT)
print("Subfolders:", [p.name for p in ROOT.iterdir() if p.is_dir()])

Using Project: Project  | Pictures folder: PicturesAlbum 
Syncing Caro's Album (29 images)
Syncing Cass's Album (59 images)
Syncing Mike’s Album (35 images)
Syncing Paolo's Album (24 images)
Syncing Melissa's Album (28 images)
Mirror complete. Downloaded 175 new files.
ROOT set to: /content/PicturesAlbum_Mirror
Subfolders: ['Mike’s Album', "Melissa's Album", "Caro's Album", "Paolo's Album", "Cass's Album"]


In [ ]:
!pip -q install pillow-heif

from pillow_heif import register_heif_opener
register_heif_opener()  # lets PIL open .heic/.heif

from pathlib import Path
import os, collections

# Point ROOT to your mirrored folder from earlier:
ROOT = Path("/content/PicturesAlbum_Mirror")  # adjust if different

# Accept common photo formats (add/remove as needed)
IMG_EXTS = {
    ".jpg",".jpeg",".png",".webp",".heic",".heif",".tif",".tiff",".gif",".bmp"
}

# Rebuild image inventory
images = []
for f in [p for p in ROOT.iterdir() if p.is_dir()]:
    for p in sorted(f.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS and os.path.getsize(p) > 0:
            images.append({"person": f.name, "path": str(p)})

print(f"Discovered {len(images)} photos across {len([p for p in ROOT.iterdir() if p.is_dir()])} folders.")

# Quick diagnostics
ext_counter = collections.Counter(Path(it["path"]).suffix.lower() for it in images)
print("By extension:", ext_counter)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 37.0 MB/s eta 0:00:00
Discovered 173 photos across 5 folders.
By extension: Counter({'.heic': 140, '.jpg': 29, '.png': 3, '.jpeg': 1})


In [ ]:
# ==== 2) OpenAI client ====
import openai
os.environ["OPENAI_API_KEY"] = input("Paste your OpenAI API key: ").strip()
openai.api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
# ==== 3) Discover teammate folders & images ====
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.heic'}
folders = [p for p in ROOT.iterdir() if p.is_dir()]
assert folders, f"No folders found under {ROOT}. Check your path."

images = []
for f in folders:
    for p in sorted(f.rglob('*')):
        if p.suffix.lower() in IMG_EXTS:
            images.append({"person": f.name, "path": str(p)})

print(f"Discovered {len(images)} images across {len(folders)} teammate folders.")

Discovered 173 images across 5 teammate folders.


In [ ]:
# ==== 4) Helper: load & base64-encode an image (downsize for speed) ====
def to_data_url(img_path, max_side=768) -> str:
    im = Image.open(img_path).convert("RGB")
    # downscale preserving aspect ratio for cheaper inference
    ratio = max(im.size)/max_side if max(im.size) > max_side else 1.0
    if ratio > 1.0:
        new_size = (int(im.size[0]/ratio), int(im.size[1]/ratio))
        im = im.resize(new_size, Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

In [ ]:
# ==== 5) Define a strict JSON schema for labels (Structured Outputs) ====
CATEGORIES = [
    "Outdoor Activities",
    "Food & Drink",
    "Cultural",
    "Night Life",
    "Concerts & Shows",
    "Casino & Gambling",
    "Attractions",
    "Sporting Events"
]

LABEL_SCHEMA = {
  "name": "photo_labels",
  "schema": {
    "type": "object",
    "properties": {
      "labels": {
        "type": "array",
        "description": "5–12 concise tags capturing scene, activity, objects, vibe.",
        "items": {"type": "string"},
        "minItems": 5,
        "maxItems": 12
      },
      "category_distribution": {
        "type": "object",
        "description": "Probability mass across the EXACT 8 categories; MUST sum to 1.0.",
        "properties": {cat: {"type": "number", "minimum": 0, "maximum": 1} for cat in CATEGORIES},
        "required": CATEGORIES,
        "additionalProperties": False
      },
      "attributes": {
        "type": "object",
        "properties": {
          "indoor_outdoor": {"type": "string", "enum": ["indoor","outdoor","mixed","unknown"]},
          "day_night": {"type": "string", "enum": ["day","night","dusk/dawn","unknown"]},
          "season_hint": {"type": "string", "enum": ["winter","spring","summer","autumn","unknown"]}
        },
        "required": ["indoor_outdoor","day_night","season_hint"],
        "additionalProperties": False
      }
    },
    "required": ["labels","category_distribution","attributes"],
    "additionalProperties": False
  }
}

SYSTEM_PROMPT = (
  "You are labeling personal photos for travel preference modeling.\n"
  "Return STRICT JSON matching the provided schema. Never include prose. \n"
  "Prefer compact, generalizable tags (e.g., 'hiking trail', 'alpine lake', 'sushi', 'museum', 'beach', 'friends hangout').\n"
  "Distribute probability mass across EXACTLY these 8 categories so they SUM TO 1.0:\n"
  f"{', '.join(CATEGORIES)}.\n"
  "If uncertain, distribute softly but still sum to 1.00. Prefer compact, generizable tags."
)

In [ ]:
# ==== 6) Vision labeling call (Responses API, image input + Structured Outputs) ====
# Docs: images & vision + structured outputs + responses API
# https://platform.openai.com/docs/guides/images-vision
# https://platform.openai.com/docs/guides/migrate-to-responses
# https://platform.openai.com/docs/guides/structured-outputs

def label_photo(data_url: str) -> Dict[str, Any]:
    # Using the Chat Completions style with image_url data URI (widely supported).
    # If you prefer the Responses API, adapt accordingly.
    completion = openai.chat.completions.create(
        model="gpt-4o",  # fast, multimodal; upgrade to GPT-5 for best quality if desired
        messages=[{
            "role": "system",
            "content": SYSTEM_PROMPT
        },{
            "role": "user",
            "content": [
                {"type": "text", "text": "Label this image following the schema."},
                {"type": "image_url", "image_url": {"url": data_url}}
            ]
        }],
        response_format={"type": "json_schema", "json_schema": LABEL_SCHEMA},
        max_tokens=300
    )
    # Parse JSON result
    txt = completion.choices[0].message.content
    return json.loads(txt)

In [ ]:
# ==== 7) Run labeling over all images (rate-limit friendly, 7-category aware) ====

# Helper in case you didn't run the earlier slug() cell
import re
if "slug" not in globals():
    def slug(s): return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

rows = []
for item in tqdm(images):
    # Robust image read (handles HEIC/animated/zero-byte via to_data_url)
    data_url = to_data_url(item["path"])
    if data_url is None:
        rows.append({
            "person": item["person"],
            "path": item["path"],
            "error": "unreadable_or_unsupported"
        })
        continue

    try:
        out = label_photo(data_url)  # <- uses your schema with 7 categories

        row = {
            "person": item["person"],
            "path": item["path"],
            "labels": out["labels"],
            "indoor_outdoor": out["attributes"]["indoor_outdoor"],
            "day_night": out["attributes"]["day_night"],
            "season_hint": out["attributes"]["season_hint"],
        }

        # Add 8 category columns in a stable order
        for cat in CATEGORIES:  # e.g., ["Outdoor Activities", ...]
            val = out.get("category_distribution", {}).get(cat, 0.0)
            row[f"cat_{slug(cat)}"] = float(val)

        rows.append(row)

        # polite pause to avoid bursty rate limits (tune as needed)
        time.sleep(0.15)

    except Exception as e:
        rows.append({
            "person": item["person"],
            "path": item["path"],
            "error": str(e)
        })

df = pd.DataFrame(rows)

# Optional: order columns nicely
cat_cols = [f"cat_{slug(c)}" for c in CATEGORIES]
base_cols = ["person", "path", "labels", "indoor_outdoor", "day_night", "season_hint"]
other_cols = [c for c in df.columns if c not in base_cols + cat_cols]
ordered_cols = [c for c in base_cols + cat_cols + other_cols if c in df.columns]
df = df[ordered_cols]

df.to_csv("/content/labels_per_photo.csv", index=False)
df.head(3)

100%|██████████| 173/173 [21:09<00:00,  7.34s/it]


,person,path,labels,indoor_outdoor,day_night,season_hint,cat_outdoor_activities,cat_food_drink,cat_cultural,cat_night_life,cat_concerts_shows,cat_casino_gambling,cat_attractions,cat_sporting_events
0,Mike’s Album,/content/PicturesAlbum_Mirror/Mike’s Album/IMG...,"[beer tasting, bar, flight of beers, indoor ga...",indoor,unknown,unknown,0.00,0.7,0.1,0.1,0.0,0.0,0.05,0.05
1,Mike’s Album,/content/PicturesAlbum_Mirror/Mike’s Album/IMG...,"[Pantheon, ancient architecture, Roman buildin...",outdoor,night,unknown,0.05,0.0,0.7,0.1,0.0,0.0,0.15,0.00
2,Mike’s Album,/content/PicturesAlbum_Mirror/Mike’s Album/IMG...,"[gym, weightlifting, barbell, fitness, indoor ...",indoor,unknown,unknown,0.00,0.0,0.0,0.0,0.0,0.0,0.00,1.00


In [ ]:
# ==== Aggregations for 8 categories (dynamic, no hardcoding) ====

import re, pandas as pd

# helper(s)
if "slug" not in globals():
    def slug(s): return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

cat_cols = [f"cat_{slug(c)}" for c in CATEGORIES]  # e.g., cat_outdoor_activities, ...
slug_to_pretty = {f"cat_{slug(c)}": c for c in CATEGORIES}

# 1) Explode labels for counting
df_ok = df.dropna(subset=["labels"]).copy()
df_ok = df_ok.explode("labels")

# 2) Per teammate (folder): label frequencies
per_person = (
    df_ok.groupby(["person", "labels"])
         .size()
         .reset_index(name="count")
         .sort_values(["person", "count"], ascending=[True, False])
)

# 3) Top 10 labels per teammate
top10_per_person = per_person.groupby("person", group_keys=False).head(10)

# 4) Category means per teammate (LONG form)
cat_means_long = (
    df[["person"] + cat_cols]
      .melt("person", var_name="category_slug", value_name="prob")
      .assign(category=lambda d: d["category_slug"].map(slug_to_pretty))
      .groupby(["person", "category"], as_index=False)["prob"].mean()
      .rename(columns={"prob": "avg_prob"})
      .sort_values(["person", "avg_prob"], ascending=[True, False])
)

# 5) Category means per teammate (WIDE form, friendly column names)
cat_means_wide = (
    cat_means_long.pivot(index="person", columns="category", values="avg_prob")
                  .reset_index()
)

# 6) Global most frequent labels
global_top = (
    df_ok.groupby("labels")
         .size()
         .reset_index(name="count")
         .sort_values("count", ascending=False)
         .head(25)
)

# 7) Save results
per_person.to_csv("/content/labels_per_person_full.csv", index=False)
top10_per_person.to_csv("/content/top10_labels_per_person.csv", index=False)
cat_means_long.to_csv("/content/category_means_per_person_long.csv", index=False)
cat_means_wide.to_csv("/content/category_means_per_person_wide.csv", index=False)
global_top.to_csv("/content/global_top_labels.csv", index=False)

cat_means_wide.head(), global_top.head(10)

(category           person  Attractions  Casino & Gambling  Concerts & Shows  \
 0            Caro's Album     0.179310           0.001724          0.034483   
 1            Cass's Album     0.236441           0.000847          0.030508   
 2         Melissa's Album     0.298214           0.000000          0.028571   
 3            Mike’s Album     0.182857           0.000000          0.021429   
 4           Paolo's Album     0.295455           0.004545          0.009091   
 
 category  Cultural  Food & Drink  Night Life  Outdoor Activities  \
 0         0.306897      0.068966    0.018966            0.286207   
 1         0.157627      0.044915    0.024576            0.325424   
 2         0.216071      0.091071    0.039286            0.294643   
 3         0.174286      0.054286    0.051429            0.467143   
 4         0.281818      0.031818    0.052273            0.306818   
 
 category  Sporting Events  
 0                0.041379  
 1                0.161017  
 2             

------
---------
## SCRAPER LABELS

In [ ]:
os.environ["OPENAI_API_KEY"] = "REPLACE WITH OPEN_AI_KEY"

In [6]:
pip install -U pillow[webp]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 52.9 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pillow<12.0,>=8.0, but you have pillow 12.0.0 which is incompatible.


In [3]:
# =========================================================
# Label hero_image_url → labels_openai (smoke test first)
# =========================================================
!pip -q install "pandas>=2,<3" "tqdm>=4,<5" "tenacity>=8,<9" pillow requests "openai>=1.40,<2"

import os, io, re, base64, requests, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt
from openai import OpenAI, AuthenticationError, APIConnectionError, RateLimitError, APIError

# ---- assumes OPENAI_API_KEY is already set (you just validated it) ----
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------- CONFIG ----------
IN_PATH  = "/content/things_to_do_all (1).csv"      # <-- change to your input CSV
OUT_PATH = "/content/tx_things_with_labels.csv" # output (checkpoints here)

HERO_COL   = "hero_image_url"
TITLE_COL  = "title"
CAT_COL    = "category"
DETAIL_COL = "detail_url"

MODEL = "gpt-4o"    # use "gpt-4o" for max quality; mini is faster/cheaper
MAX_SIDE = 768           # image max side (downscaled)
BATCH_SAVE_EVERY = 50    # checkpoint freq
SMOKE_N = 5              # <-- run a tiny sample first; set to 0 to run ALL

SYSTEM_PROMPT = (
    "You are an expert image tagger. Return a short, comma-separated list of "
    "specific, objective labels capturing objects, setting, actions, colors, and notable attributes. "
    "No sentences—just tags. 8–15 tags max."
)

# ---------- Load CSV (+ resume if OUT exists) ----------
df = pd.read_csv(IN_PATH)
for col in [HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if os.path.exists(OUT_PATH):
    prev = pd.read_csv(OUT_PATH)
    df = df.merge(prev[[HERO_COL, TITLE_COL, "labels_openai", "label_error"]],
                  on=[HERO_COL, TITLE_COL], how="left", suffixes=("", "_prev"))
    if "labels_openai_prev" in df.columns and "labels_openai" not in df.columns:
        df.rename(columns={"labels_openai_prev":"labels_openai"}, inplace=True)
    if "label_error_prev" in df.columns and "label_error" not in df.columns:
        df.rename(columns={"label_error_prev":"label_error"}, inplace=True)
else:
    df["labels_openai"] = ""
    df["label_error"] = ""

def save_checkpoint(_df):
    _df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].to_csv(
        OUT_PATH, index=False
    )

# ---------- Helpers ----------
def is_http_url(s: str) -> bool:
    return isinstance(s, str) and re.match(r"^https?://", s.strip()) is not None

def image_to_data_url(url: str, max_side=MAX_SIDE, timeout=15) -> str | None:
    """Download → resize → JPEG → base64 data URL. None on failure."""
    try:
        r = requests.get(url, timeout=timeout, stream=True, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        raw = io.BytesIO(r.content)
        im = Image.open(raw)
        im = ImageOps.exif_transpose(im)
        if getattr(im, "is_animated", False):
            im.seek(0)
        if im.mode not in ("RGB", "L"):
            im = im.convert("RGB")
        w, h = im.size
        s = max(w, h) / max_side if max(w, h) > max_side else 1.0
        if s > 1.0:
            im = im.resize((int(w/s), int(h/s)), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=88, optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except (requests.RequestException, UnidentifiedImageError, OSError):
        return None

@retry(wait=wait_exponential(min=1, max=12), stop=stop_after_attempt(4))
def tag_image(data_url: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "input_text", "text": "Tag this image with concise, informative labels."},
                {"type": "input_image", "image": data_url},
            ]},
        ],
        max_output_tokens=200,
    )
    return (resp.output_text or "").strip()

# ---------- Choose rows to run ----------
to_run = df.index[(df["labels_openai"].astype(str).str.len()==0) & (df["label_error"].astype(str)=="")]
if SMOKE_N > 0:
    to_run = to_run[:SMOKE_N]
    print(f"SMOKE TEST: processing {len(to_run)} rows. Set SMOKE_N=0 to run ALL after verifying.")

ok, fail, since_save = 0, 0, 0

for i in tqdm(to_run, desc="Labeling"):
    url = str(df.at[i, HERO_COL]).strip()
    if not is_http_url(url):
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "invalid_url"
        fail += 1
        continue

    data_url = image_to_data_url(url)
    if not data_url:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "download_or_decode_failed"
        fail += 1
        continue

    try:
        tags = tag_image(data_url)
        if not tags or ("," not in tags and len(tags.split()) > 6):
            raise ValueError("empty_or_non_tag_output")
        df.at[i, "labels_openai"] = tags
        df.at[i, "label_error"] = ""
        ok += 1
    except AuthenticationError:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "auth_error"
        save_checkpoint(df)
        raise SystemExit("❌ Authentication error mid-run. Re-check OPENAI_API_KEY.")
    except (APIConnectionError, RateLimitError, APIError) as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"api_error:{type(e).__name__}"
        fail += 1
    except Exception as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"runtime:{type(e).__name__}"
        fail += 1

    since_save += 1
    if since_save >= BATCH_SAVE_EVERY:
        save_checkpoint(df); since_save = 0

# final save
save_checkpoint(df)
print(f"Done. ✅ {ok} labeled | ❌ {fail} failed | Output → {OUT_PATH}")

# Quick sanity view
df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].head(10)

SMOKE TEST: processing 0 rows. Set SMOKE_N=0 to run ALL after verifying.


Labeling: 0it [00:00, ?it/s]

Done. ✅ 0 labeled | ❌ 0 failed | Output → /content/tx_things_with_labels.csv


,hero_image_url,title,category,detail_url,labels_openai,label_error
0,https://cdn-v2.theculturetrip.com/1200x630/wp-...,14 Things To Do And See In Downtown Houston,SEE AND DO,https://theculturetrip.com/articles/10-locatio...,NaN,download_or_decode_failed
1,https://cdn-v2.theculturetrip.com/1200x630/wp-...,6 Sensational Places to See Fall Foliage in Texas,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/where-to-s...,NaN,download_or_decode_failed
2,https://cdn-v2.theculturetrip.com/1200x630/wp-...,8 Cool Things to Do in Dallas at Night,SEE AND DO,https://theculturetrip.com/articles/8-cool-thi...,NaN,download_or_decode_failed
3,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Quiet Escapes In Houston, Texas",SEE AND DO,https://theculturetrip.com/articles/10-quiet-e...,NaN,download_or_decode_failed
4,https://cdn-v2.theculturetrip.com/1200x630/wp-...,Best Weekend Getaways From Dallas & Fort Worth,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/10-best-we...,NaN,download_or_decode_failed
5,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 8 Most Beautiful Churches And Cathedrals I...,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/the-8-most...,NaN,NaN
6,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 13 Most Beautiful Places in Texas to Add t...,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/the-13-mos...,NaN,NaN
7,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"The 9 Best Things to Do In Deep Ellum, Dallas",SEE AND DO,https://theculturetrip.com/articles/10-fun-thi...,NaN,NaN
8,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"10 Awesome Things To Do And See In Harlingen, ...",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN
9,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Top 10 Things To Do In El Paso, Texas",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN


In [ ]:
# =========================
# Robust image downloader (CDN-friendly) + pipeline patch
# =========================
!pip -q install requests pillow

import os, io, re, base64, time, requests, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from urllib.parse import urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Build a retrying session (handles 429/5xx + backoff) ---
def build_session():
    s = requests.Session()
    retry = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=0.6,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"])
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    return s

SESSION = build_session()

# --- Pick a referer automatically from the article page or host ---
def pick_referer(image_url: str, detail_url: str | None = None) -> str | None:
    if detail_url and re.match(r"^https?://", str(detail_url)):
        return detail_url  # best referer is the page itself
    host = urlparse(image_url).hostname or ""
    # site-specific fallbacks
    if host.endswith("theculturetrip.com"):
        return "https://theculturetrip.com/"
    return None  # no referer

# --- CDN-friendly downloader: GET -> PIL -> JPEG -> base64 ---
def image_to_data_url(url: str, referer: str | None = None,
                      max_side: int = 768,
                      timeouts=((5,12),(5,20),(5,30))) -> str | None:
    """
    Downloads an image with robust headers, optional referer, retries via session,
    resizes to max_side, converts to JPEG, returns data URL. None on failure.
    timeouts is a sequence of (connect_timeout, read_timeout) tuples tried in order.
    """
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "image/avif,image/webp,image/*,*/*;q=0.8",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    }
    if referer:
        headers["Referer"] = referer

    # Try multiple read timeouts
    for (tc, tr) in timeouts:
        try:
            r = SESSION.get(url, headers=headers, timeout=(tc, tr), stream=True, allow_redirects=True)
            r.raise_for_status()
            content = r.content
            im = Image.open(io.BytesIO(content))
            im = ImageOps.exif_transpose(im)
            if getattr(im, "is_animated", False):
                im.seek(0)
            if im.mode not in ("RGB", "L"):
                im = im.convert("RGB")
            w, h = im.size
            s = max(w, h) / max_side if max(w, h) > max_side else 1.0
            if s > 1.0:
                im = im.resize((int(w/s), int(h/s)), Image.LANCZOS)
            buf = io.BytesIO()
            im.save(buf, format="JPEG", quality=88, optimize=True)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}"
        except (requests.Timeout, requests.ReadTimeout):
            # try the next (longer) timeout
            continue
        except (requests.RequestException, UnidentifiedImageError, OSError):
            return None
    return None

# --- Test the exact URL you sent (should now succeed or fail fast) ---
TEST_URL = "https://cdn-v2.theculturetrip.com/1200x630/wp-content/uploads/2024/07/shutterstock_2456508113-1.webp"
du = image_to_data_url(TEST_URL, referer=pick_referer(TEST_URL, "https://theculturetrip.com/"))
print("Downloader:", "OK" if du else "FAILED")

# =========================
# Plug into your labeling loop
# =========================
# Assumes df exists with columns hero_image_url, title, category, detail_url
# and that 'client' + tag_with_chat_completions() are already defined (as in previous cell)

SMOKE_N = 5  # keep this small until you see results

def label_row(i):
    url = str(df.at[i, "hero_image_url"]).strip()
    ref = pick_referer(url, str(df.at[i, "detail_url"]))
    if not re.match(r"^https?://", url):
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "invalid_url"
        return False
    data_url = image_to_data_url(url, referer=ref)
    if not data_url:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "download_timeout_or_decode_failed"
        return False
    try:
        tags = tag_with_chat_completions(data_url)  # your chat completions tagger (gpt-4o)
        if not tags or ("," not in tags and len(tags.split()) > 6):
            raise ValueError("empty_or_non_tag_output")
        df.at[i, "labels_openai"] = tags
        df.at[i, "label_error"] = ""
        return True
    except Exception as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"tagger:{type(e).__name__}"
        return False

# Choose a few rows to test first
todo = df.index[(df["labels_openai"].astype(str).str.len()==0) & (df["label_error"].astype(str)=="")]
todo = todo[:SMOKE_N]
print("Smoke-test rows:", len(todo))

ok = 0
for i in todo:
    ok += 1 if label_row(i) else 0

print(f"Smoke test done. OK={ok}/{len(todo)}")
# Save a quick preview CSV so you can inspect
df[["hero_image_url","title","category","detail_url","labels_openai","label_error"]].head(10)

Downloader: FAILED
Smoke-test rows: 0
Smoke test done. OK=0/0


,hero_image_url,title,category,detail_url,labels_openai,label_error
0,https://cdn-v2.theculturetrip.com/1200x630/wp-...,14 Things To Do And See In Downtown Houston,SEE AND DO,https://theculturetrip.com/articles/10-locatio...,NaN,RetryError[<Future at 0x7bd7b24aa810 state=fin...
1,https://cdn-v2.theculturetrip.com/1200x630/wp-...,6 Sensational Places to See Fall Foliage in Texas,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/where-to-s...,NaN,RetryError[<Future at 0x7bd7b24aacf0 state=fin...
2,https://cdn-v2.theculturetrip.com/1200x630/wp-...,8 Cool Things to Do in Dallas at Night,SEE AND DO,https://theculturetrip.com/articles/8-cool-thi...,NaN,RetryError[<Future at 0x7bd7b251b650 state=fin...
3,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Quiet Escapes In Houston, Texas",SEE AND DO,https://theculturetrip.com/articles/10-quiet-e...,NaN,RetryError[<Future at 0x7bd7b24a6450 state=fin...
4,https://cdn-v2.theculturetrip.com/1200x630/wp-...,Best Weekend Getaways From Dallas & Fort Worth,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/10-best-we...,NaN,RetryError[<Future at 0x7bd7b2404080 state=fin...
5,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 8 Most Beautiful Churches And Cathedrals I...,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/the-8-most...,NaN,RetryError[<Future at 0x7bd7b23fb980 state=fin...
6,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 13 Most Beautiful Places in Texas to Add t...,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/the-13-mos...,NaN,RetryError[<Future at 0x7bd7b2406ab0 state=fin...
7,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"The 9 Best Things to Do In Deep Ellum, Dallas",SEE AND DO,https://theculturetrip.com/articles/10-fun-thi...,NaN,RetryError[<Future at 0x7bd7b24a6ed0 state=fin...
8,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"10 Awesome Things To Do And See In Harlingen, ...",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,RetryError[<Future at 0x7bd7b23bc410 state=fin...
9,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Top 10 Things To Do In El Paso, Texas",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,RetryError[<Future at 0x7bd7b23f82f0 state=fin...


In [4]:
pip install imageio

In [5]:
# =========================================================
# Label hero_image_url → labels_openai (smoke test first)
# =========================================================
!pip -q install "pandas>=2,<3" "tqdm>=4,<5" "tenacity>=8,<9" pillow requests "openai>=1.40,<2"

import os, io, re, base64, requests, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt
from openai import OpenAI, AuthenticationError, APIConnectionError, RateLimitError, APIError

# ---- assumes OPENAI_API_KEY is already set (you just validated it) ----
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------- CONFIG ----------
IN_PATH  = "/content/things_to_do_all (1).csv"      # <-- change to your input CSV
OUT_PATH = "/content/tx_things_with_labels.csv" # output (checkpoints here)

HERO_COL   = "hero_image_url"
TITLE_COL  = "title"
CAT_COL    = "category"
DETAIL_COL = "detail_url"

MODEL = "gpt-4o"    # use "gpt-4o" for max quality; mini is faster/cheaper
MAX_SIDE = 768           # image max side (downscaled)
BATCH_SAVE_EVERY = 50    # checkpoint freq
SMOKE_N = 5              # <-- run a tiny sample first; set to 0 to run ALL

SYSTEM_PROMPT = (
    "You are an expert image tagger. Return a short, comma-separated list of "
    "specific, objective labels capturing objects, setting, actions, colors, and notable attributes. "
    "No sentences—just tags. 8–15 tags max."
)

# ---------- Load CSV (+ resume if OUT exists) ----------
df = pd.read_csv(IN_PATH)
for col in [HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if os.path.exists(OUT_PATH):
    prev = pd.read_csv(OUT_PATH)
    df = df.merge(prev[[HERO_COL, TITLE_COL, "labels_openai", "label_error"]],
                  on=[HERO_COL, TITLE_COL], how="left", suffixes=("", "_prev"))
    if "labels_openai_prev" in df.columns and "labels_openai" not in df.columns:
        df.rename(columns={"labels_openai_prev":"labels_openai"}, inplace=True)
    if "label_error_prev" in df.columns and "label_error" not in df.columns:
        df.rename(columns={"label_error_prev":"label_error"}, inplace=True)
else:
    df["labels_openai"] = ""
    df["label_error"] = ""

def save_checkpoint(_df):
    _df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].to_csv(
        OUT_PATH, index=False
    )

# ---------- Helpers ----------
def is_http_url(s: str) -> bool:
    return isinstance(s, str) and re.match(r"^https?://", s.strip()) is not None

# =========================================================
# Label hero_image_url → labels_openai (smoke test first)
# =========================================================
!pip -q install "pandas>=2,<3" "tqdm>=4,<5" "tenacity>=8,<9" pillow requests "openai>=1.40,<2"

import os, io, re, base64, requests, pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt
from openai import OpenAI, AuthenticationError, APIConnectionError, RateLimitError, APIError

# ---- assumes OPENAI_API_KEY is already set (you just validated it) ----
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------- CONFIG ----------
IN_PATH  = "/content/things_to_do_all (1).csv"      # <-- change to your input CSV
OUT_PATH = "/content/tx_things_with_labels.csv" # output (checkpoints here)

HERO_COL   = "hero_image_url"
TITLE_COL  = "title"
CAT_COL    = "category"
DETAIL_COL = "detail_url"

MODEL = "gpt-4o"    # use "gpt-4o" for max quality; mini is faster/cheaper
MAX_SIDE = 768           # image max side (downscaled)
BATCH_SAVE_EVERY = 50    # checkpoint freq
SMOKE_N = 5              # <-- run a tiny sample first; set to 0 to run ALL

SYSTEM_PROMPT = (
    "You are an expert image tagger. Return a short, comma-separated list of "
    "specific, objective labels capturing objects, setting, actions, colors, and notable attributes. "
    "No sentences—just tags. 8–15 tags max."
)

# ---------- Load CSV (+ resume if OUT exists) ----------
df = pd.read_csv(IN_PATH)
for col in [HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if os.path.exists(OUT_PATH):
    prev = pd.read_csv(OUT_PATH)
    df = df.merge(prev[[HERO_COL, TITLE_COL, "labels_openai", "label_error"]],
                  on=[HERO_COL, TITLE_COL], how="left", suffixes=("", "_prev"))
    if "labels_openai_prev" in df.columns and "labels_openai" not in df.columns:
        df.rename(columns={"labels_openai_prev":"labels_openai"}, inplace=True)
    if "label_error_prev" in df.columns and "label_error" not in df.columns:
        df.rename(columns={"label_error_prev":"label_error"}, inplace=True)
else:
    df["labels_openai"] = ""
    df["label_error"] = ""

def save_checkpoint(_df):
    _df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].to_csv(
        OUT_PATH, index=False
    )

# ---------- Helpers ----------
def is_http_url(s: str) -> bool:
    return isinstance(s, str) and re.match(r"^https?://", s.strip()) is not None

def image_to_data_url(url: str, max_side=MAX_SIDE, timeout=15) -> str | None:
    """Download → resize → JPEG → base64 data URL. None on failure."""
    try:
        r = requests.get(url, timeout=timeout, stream=True, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        raw = io.BytesIO(r.content)
        im = Image.open(raw)
        im = ImageOps.exif_transpose(im)
        if getattr(im, "is_animated", False):
            im.seek(0)
        if im.mode not in ("RGB", "L"):
            im = im.convert("RGB")
        w, h = im.size
        s = max(w, h) / max_side if max(w, h) > max_side else 1.0
        if s > 1.0:
            im = im.resize((int(w/s), int(h/s)), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=88, optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except (requests.RequestException, UnidentifiedImageError, OSError):
        return None

@retry(wait=wait_exponential(min=1, max=12), stop=stop_after_attempt(4))
def tag_image(data_url: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "input_text", "text": "Tag this image with concise, informative labels."},
                {"type": "input_image", "image": data_url},
            ]},
        ],
        max_output_tokens=200,
    )
    return (resp.output_text or "").strip()

# ---------- Choose rows to run ----------
to_run = df.index[(df["labels_openai"].astype(str).str.len()==0) & (df["label_error"].astype(str)=="")]
if SMOKE_N > 0:
    to_run = to_run[:SMOKE_N]
    print(f"SMOKE TEST: processing {len(to_run)} rows. Set SMOKE_N=0 to run ALL after verifying.")

ok, fail, since_save = 0, 0, 0

for i in tqdm(to_run, desc="Labeling"):
    url = str(df.at[i, HERO_COL]).strip()
    if not is_http_url(url):
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "invalid_url"
        fail += 1
        continue

    data_url = image_to_data_url(url)
    if not data_url:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "download_or_decode_failed"
        fail += 1
        continue

    try:
        tags = tag_image(data_url)
        if not tags or ("," not in tags and len(tags.split()) > 6):
            raise ValueError("empty_or_non_tag_output")
        df.at[i, "labels_openai"] = tags
        df.at[i, "label_error"] = ""
        ok += 1
    except AuthenticationError:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "auth_error"
        save_checkpoint(df)
        raise SystemExit("❌ Authentication error mid-run. Re-check OPENAI_API_KEY.")
    except (APIConnectionError, RateLimitError, APIError) as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"api_error:{type(e).__name__}"
        fail += 1
    except Exception as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"runtime:{type(e).__name__}"
        fail += 1

    since_save += 1
    if since_save >= BATCH_SAVE_EVERY:
        save_checkpoint(df); since_save = 0

# final save
save_checkpoint(df)
print(f"Done. ✅ {ok} labeled | ❌ {fail} failed | Output → {OUT_PATH}")

# Quick sanity view
df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].head(10)

@retry(wait=wait_exponential(min=1, max=12), stop=stop_after_attempt(4))
def tag_image(data_url: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "input_text", "text": "Tag this image with concise, informative labels."},
                {"type": "input_image", "image": data_url},
            ]},
        ],
        max_output_tokens=200,
    )
    return (resp.output_text or "").strip()

# ---------- Choose rows to run ----------
to_run = df.index[(df["labels_openai"].astype(str).str.len()==0) & (df["label_error"].astype(str)=="")]
if SMOKE_N > 0:
    to_run = to_run[:SMOKE_N]
    print(f"SMOKE TEST: processing {len(to_run)} rows. Set SMOKE_N=0 to run ALL after verifying.")

ok, fail, since_save = 0, 0, 0

for i in tqdm(to_run, desc="Labeling"):
    url = str(df.at[i, HERO_COL]).strip()
    if not is_http_url(url):
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "invalid_url"
        fail += 1
        continue

    data_url = image_to_data_url(url)
    if not data_url:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "download_or_decode_failed"
        fail += 1
        continue

    try:
        tags = tag_image(data_url)
        if not tags or ("," not in tags and len(tags.split()) > 6):
            raise ValueError("empty_or_non_tag_output")
        df.at[i, "labels_openai"] = tags
        df.at[i, "label_error"] = ""
        ok += 1
    except AuthenticationError:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "auth_error"
        save_checkpoint(df)
        raise SystemExit("❌ Authentication error mid-run. Re-check OPENAI_API_KEY.")
    except (APIConnectionError, RateLimitError, APIError) as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"api_error:{type(e).__name__}"
        fail += 1
    except Exception as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"runtime:{type(e).__name__}"
        fail += 1

    since_save += 1
    if since_save >= BATCH_SAVE_EVERY:
        save_checkpoint(df); since_save = 0

# final save
save_checkpoint(df)
print(f"Done. ✅ {ok} labeled | ❌ {fail} failed | Output → {OUT_PATH}")

# Quick sanity view
df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].head(10)

SMOKE TEST: processing 0 rows. Set SMOKE_N=0 to run ALL after verifying.


Labeling: 0it [00:00, ?it/s]


Done. ✅ 0 labeled | ❌ 0 failed | Output → /content/tx_things_with_labels.csv
SMOKE TEST: processing 0 rows. Set SMOKE_N=0 to run ALL after verifying.


Labeling: 0it [00:00, ?it/s]

Done. ✅ 0 labeled | ❌ 0 failed | Output → /content/tx_things_with_labels.csv


,hero_image_url,title,category,detail_url,labels_openai,label_error
0,https://cdn-v2.theculturetrip.com/1200x630/wp-...,14 Things To Do And See In Downtown Houston,SEE AND DO,https://theculturetrip.com/articles/10-locatio...,NaN,download_or_decode_failed
1,https://cdn-v2.theculturetrip.com/1200x630/wp-...,6 Sensational Places to See Fall Foliage in Texas,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/where-to-s...,NaN,download_or_decode_failed
2,https://cdn-v2.theculturetrip.com/1200x630/wp-...,8 Cool Things to Do in Dallas at Night,SEE AND DO,https://theculturetrip.com/articles/8-cool-thi...,NaN,download_or_decode_failed
3,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Quiet Escapes In Houston, Texas",SEE AND DO,https://theculturetrip.com/articles/10-quiet-e...,NaN,download_or_decode_failed
4,https://cdn-v2.theculturetrip.com/1200x630/wp-...,Best Weekend Getaways From Dallas & Fort Worth,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/10-best-we...,NaN,download_or_decode_failed
5,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 8 Most Beautiful Churches And Cathedrals I...,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/the-8-most...,NaN,NaN
6,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 13 Most Beautiful Places in Texas to Add t...,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/the-13-mos...,NaN,NaN
7,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"The 9 Best Things to Do In Deep Ellum, Dallas",SEE AND DO,https://theculturetrip.com/articles/10-fun-thi...,NaN,NaN
8,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"10 Awesome Things To Do And See In Harlingen, ...",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN
9,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Top 10 Things To Do In El Paso, Texas",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN


In [7]:
import requests, io
from PIL import Image

url = "https://cdn-v2.theculturetrip.com/1200x630/wp-content/uploads/2024/07/shutterstock_2456508113-1.webp"

try:
    r = requests.get(
        url,
        stream=True,
        timeout=10,  # seconds
        headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Accept": "image/webp,image/*,*/*;q=0.8",
            "Referer": "https://www.theculturetrip.com/"
        }
    )
    print("Status:", r.status_code, "Length:", len(r.content))
    print("Content type:", r.headers.get("Content-Type", "N/A"))

    im = Image.open(io.BytesIO(r.content))
    print("✅ Decoded OK:", im.format, im.size)
except Exception as e:
    print("❌ Error:", e)



❌ Error: HTTPSConnectionPool(host='cdn-v2.theculturetrip.com', port=443): Read timed out. (read timeout=10)


In [4]:
# =========================================================
# Label hero_image_url → labels_openai (using direct URLs)
# =========================================================
!pip -q install "pandas>=2,<3" "tqdm>=4,<5" "tenacity>=8,<9" "openai>=1.40,<2"

import os, re, pandas as pd
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt
from openai import OpenAI, AuthenticationError, APIConnectionError, RateLimitError, APIError

# ---- assumes OPENAI_API_KEY is already set ----
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------- CONFIG ----------
IN_PATH  = "/content/things_to_do_all (1).csv"
OUT_PATH = "/content/tx_things_with_labels.csv"

HERO_COL   = "hero_image_url"
TITLE_COL  = "title"
CAT_COL    = "category"
DETAIL_COL = "detail_url"

MODEL = "gpt-4o"    # or "gpt-4o-mini" for cheaper testing
BATCH_SAVE_EVERY = 50
SMOKE_N = 5          # set to 0 to process all

SYSTEM_PROMPT = (
    "You are an expert image tagger. Return a short, comma-separated list of "
    "specific, objective labels capturing objects, setting, actions, colors, and notable attributes. "
    "No sentences—just tags. 8–15 tags max."
)

# ---------- Load CSV (+ resume if OUT exists) ----------
df = pd.read_csv(IN_PATH)
for col in [HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if os.path.exists(OUT_PATH):
    prev = pd.read_csv(OUT_PATH)
    df = df.merge(prev[[HERO_COL, TITLE_COL, "labels_openai", "label_error"]],
                  on=[HERO_COL, TITLE_COL], how="left", suffixes=("", "_prev"))
    if "labels_openai_prev" in df.columns and "labels_openai" not in df.columns:
        df.rename(columns={"labels_openai_prev":"labels_openai"}, inplace=True)
    if "label_error_prev" in df.columns and "label_error" not in df.columns:
        df.rename(columns={"label_error_prev":"label_error"}, inplace=True)
else:
    df["labels_openai"] = ""
    df["label_error"] = ""

def save_checkpoint(_df):
    _df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].to_csv(
        OUT_PATH, index=False
    )

# ---------- Helpers ----------
def is_http_url(s: str) -> bool:
    return isinstance(s, str) and re.match(r"^https?://", s.strip()) is not None

@retry(wait=wait_exponential(min=1, max=12), stop=stop_after_attempt(4))
def tag_image_direct(url: str) -> str:
    """Send image URL directly to OpenAI Vision."""
    resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "input_text", "text": "Tag this image with concise, informative labels."},
                {"type": "input_image", "image_url": url},
            ]},
        ],
        max_output_tokens=200,
    )
    return (resp.output_text or "").strip()

# ---------- Choose rows to run ----------
to_run = df.index[(df["labels_openai"].astype(str).str.len()==0) & (df["label_error"].astype(str)=="")]
if SMOKE_N > 0:
    to_run = to_run[:SMOKE_N]
    print(f"SMOKE TEST: processing {len(to_run)} rows. Set SMOKE_N=0 to run ALL after verifying.")

ok, fail, since_save = 0, 0, 0

for i in tqdm(to_run, desc="Labeling"):
    url = str(df.at[i, HERO_COL]).strip()
    if not is_http_url(url):
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "invalid_url"
        fail += 1
        continue
    print(f"Processing row {i}, URL={url[:60]}...", flush=True)
    try:
        print("Calling tag_image_direct()")
        tags = tag_image_direct(url)
        if not tags or ("," not in tags and len(tags.split()) > 6):
            raise ValueError("empty_or_non_tag_output")
        df.at[i, "labels_openai"] = tags
        df.at[i, "label_error"] = ""
        ok += 1
    except AuthenticationError:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = "auth_error"
        save_checkpoint(df)
        raise SystemExit("❌ Authentication error mid-run. Re-check OPENAI_API_KEY.")
    except (APIConnectionError, RateLimitError, APIError) as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"api_error:{type(e).__name__}"
        fail += 1
    except Exception as e:
        df.at[i, "labels_openai"] = ""
        df.at[i, "label_error"] = f"runtime:{type(e).__name__}"
        fail += 1

    since_save += 1
    if since_save >= BATCH_SAVE_EVERY:
        save_checkpoint(df); since_save = 0

# final save
save_checkpoint(df)
print(f"Done. ✅ {ok} labeled | ❌ {fail} failed | Output → {OUT_PATH}")

# Quick sanity view
df[[HERO_COL, TITLE_COL, CAT_COL, DETAIL_COL, "labels_openai", "label_error"]].head(10)



SMOKE TEST: processing 0 rows. Set SMOKE_N=0 to run ALL after verifying.


Labeling: 0it [00:00, ?it/s]

Done. ✅ 0 labeled | ❌ 0 failed | Output → /content/tx_things_with_labels.csv


,hero_image_url,title,category,detail_url,labels_openai,label_error
0,https://cdn-v2.theculturetrip.com/1200x630/wp-...,14 Things To Do And See In Downtown Houston,SEE AND DO,https://theculturetrip.com/articles/10-locatio...,NaN,download_or_decode_failed
1,https://cdn-v2.theculturetrip.com/1200x630/wp-...,6 Sensational Places to See Fall Foliage in Texas,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/where-to-s...,NaN,download_or_decode_failed
2,https://cdn-v2.theculturetrip.com/1200x630/wp-...,8 Cool Things to Do in Dallas at Night,SEE AND DO,https://theculturetrip.com/articles/8-cool-thi...,NaN,download_or_decode_failed
3,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Quiet Escapes In Houston, Texas",SEE AND DO,https://theculturetrip.com/articles/10-quiet-e...,NaN,download_or_decode_failed
4,https://cdn-v2.theculturetrip.com/1200x630/wp-...,Best Weekend Getaways From Dallas & Fort Worth,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/10-best-we...,NaN,download_or_decode_failed
5,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 8 Most Beautiful Churches And Cathedrals I...,RECOMMENDATIONS - ATTRACTIONS,https://theculturetrip.com/articles/the-8-most...,NaN,NaN
6,https://cdn-v2.theculturetrip.com/1200x630/wp-...,The 13 Most Beautiful Places in Texas to Add t...,RECOMMENDATIONS - OUTDOORS,https://theculturetrip.com/articles/the-13-mos...,NaN,NaN
7,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"The 9 Best Things to Do In Deep Ellum, Dallas",SEE AND DO,https://theculturetrip.com/articles/10-fun-thi...,NaN,NaN
8,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"10 Awesome Things To Do And See In Harlingen, ...",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN
9,https://cdn-v2.theculturetrip.com/1200x630/wp-...,"Top 10 Things To Do In El Paso, Texas",SEE AND DO,https://theculturetrip.com/articles/the-top-10...,NaN,NaN


In [12]:
print("✅ Using tag_image_direct:", 'tag_image_direct' in globals())

✅ Using tag_image_direct: True
